# Decisions with SAP-RPT-1.6: typed questions with calibrated probabilities

The RPT Playground offers a **Decision API** (`POST /api/systemone`). Instead of a table you send:

* a `state`: the thing to judge (a text, a JSON object or a chat log);
* typed `questions`: `noul` (yes/no), `choice` (one of several options) or `score` (ordinal level);
* labeled `examples` (2-2,048) that RPT learns from in context.

and get back a probability distribution per question, for example triaging a customer message by *is it about billing?*, *what is the tone?* and *how urgent is it?*.

This notebook shows how to get the same capability on **SAP AI Core**, where only the tabular `/predict` endpoint exists:

1. Check whether an AI Core RPT-1.6 deployment accepts the Playground body (it does not).
2. Translate the decision request into an RPT table with `rpt_decisions.py` and call `/predict`.
3. Read the answers in the Playground's response shape.
4. Compare two table encodings, batch many states in one call, and use structured states.

> The translation in `rpt_decisions.py` is our own design, built from the Playground's documented behaviour. It is not SAP's internal Playground implementation, so validate it on labeled data from your own use case.

All example messages in this notebook are short illustrative texts (taken from or modeled on the Playground documentation), not customer data.

## 1. Setup

In [1]:
import json

import pandas as pd
from dotenv import find_dotenv, load_dotenv

from rpt_client_api import RPTClientAPI
from rpt_decisions import build_decision_payload, decide, parse_decision_answers, validate_decision_request

_ = load_dotenv(find_dotenv(usecwd=True))
client = RPTClientAPI(model_name="sap-rpt-1.6")  # the Playground's default decision model
pd.set_option("display.max_colwidth", 80)

## 2. The decision request

The request below follows the Playground body: three questions of the three types and five labeled examples. Answer formats per type:

| Type | Example answer | Returned |
|---|---|---|
| `noul` | `true` / `false` | probability of `true` |
| `choice` | option name, e.g. `"angry"` | winning option, distribution, confidence |
| `score` | level index as string, e.g. `"2"` | expected level, distribution, confidence |

**Label demonstration rule:** every option or level must appear in at least one example; RPT can only predict labels it has seen.

In [2]:
triage = {
    "model": "sap-rpt-1.6",
    "state": "I was charged $99 instead of $49 on my last bill, please fix this!",
    "questions": {
        "billing": {"type": "noul", "instructions": "Is this message about billing?"},
        "tone": {
            "type": "choice",
            "instructions": "What is the tone of the message?",
            "criteria": {
                "calm": "Polite, positive, or appreciative",
                "angry": "Hostile, frustrated, or threatening",
                "neutral": "Factual, neither positive nor negative",
            },
        },
        "priority": {
            "type": "score",
            "instructions": "How urgently does this need attention?",
            "criteria": ["can wait", "this week", "today"],
        },
    },
    "examples": [
        {"state": "Thanks so much, the new feature is great!", "answers": {"billing": False, "tone": "calm", "priority": "0"}},
        {"state": "You billed my card three times, refund me!", "answers": {"billing": True, "tone": "angry", "priority": "2"}},
        {"state": "When will the next release be available?", "answers": {"billing": False, "tone": "neutral", "priority": "1"}},
        {"state": "My invoice total looks wrong this month.", "answers": {"billing": True, "tone": "neutral", "priority": "1"}},
        {"state": "Third time I've been overcharged. Unacceptable.", "answers": {"billing": True, "tone": "angry", "priority": "2"}},
    ],
}

## 3. Does AI Core accept the Playground body?

We send it unchanged to the RPT-1.6 deployment. The `/predict` schema is strict: it requires `prediction_config` and rejects unknown fields. (We also tried `/systemone`, `/decision` and `/decisions` on the deployment; they return HTTP 404.)

In [3]:
import requests

# predict_json refuses a body without prediction_config before sending it,
# so this check posts the unchanged Playground body with plain requests.
probe = requests.post(
    client.deployment_url + "/predict",
    headers=client._headers(content_type="application/json"),
    json=triage,
    timeout=60,
)
print("HTTP", probe.status_code, "| status:", probe.json().get("status"))
for issue in probe.json().get("detail", []):
    print(" ", issue["loc"], "->", issue["msg"])

HTTP 422 | status: {'code': 2, 'message': 'Invalid input'}
  ['prediction_config'] -> Field required
  ['model'] -> Extra inputs are not permitted
  ['state'] -> Extra inputs are not permitted
  ['questions'] -> Extra inputs are not permitted
  ['examples'] -> Extra inputs are not permitted


`state`, `questions`, `examples` and `model` are "extra inputs". So on AI Core the decision has to be expressed as a table.

## 4. Translate the decision into an RPT table

`build_decision_payload` builds one RPT request:

| Table part | Built from |
|---|---|
| index column `row_id` | `example-0`, ... and `query-0`, ... |
| feature column(s) | the state: a text becomes `state`; a JSON object becomes one column per key; a list (chat log) becomes JSON text |
| one target column per question | example rows hold the answer, query rows hold `[PREDICT]` |
| `prediction_config` | every question is a `classification` target with `top_k` = number of options, so the **full distribution** comes back |

With the default **`described`** encoding the question `instructions` become the column names, and the option descriptions go into the cell values (`"angry: Hostile, frustrated, or threatening"`, score levels as their labels). RPT reads column names and values semantically, so this gives it the meaning of each option, not just an opaque key.

In [4]:
payload = build_decision_payload(triage)
print(json.dumps(payload["prediction_config"], indent=2))
pd.DataFrame(payload["rows"])

{
  "target_columns": [
    {
      "name": "Is this message about billing?",
      "prediction_placeholder": "[PREDICT]",
      "task_type": "classification",
      "top_k": 2
    },
    {
      "name": "What is the tone of the message?",
      "prediction_placeholder": "[PREDICT]",
      "task_type": "classification",
      "top_k": 3
    },
    {
      "name": "How urgently does this need attention?",
      "prediction_placeholder": "[PREDICT]",
      "task_type": "classification",
      "top_k": 3
    }
  ]
}


,row_id,state,Is this message about billing?,What is the tone of the message?,How urgently does this need attention?
0,example-0,"Thanks so much, the new feature is great!",no,"calm: Polite, positive, or appreciative",can wait
1,example-1,"You billed my card three times, refund me!",yes,"angry: Hostile, frustrated, or threatening",today
2,example-2,When will the next release be available?,no,"neutral: Factual, neither positive nor negative",this week
3,example-3,My invoice total looks wrong this month.,yes,"neutral: Factual, neither positive nor negative",this week
4,example-4,Third time I've been overcharged. Unacceptable.,yes,"angry: Hostile, frustrated, or threatening",today
5,query-0,"I was charged $99 instead of $49 on my last bill, please fix this!",[PREDICT],[PREDICT],[PREDICT]


## 5. Decide

`decide` sends the table to `/predict` in one joint call and maps the response back with `parse_decision_answers`. The result has the Playground's answer shape:

In [5]:
result = decide(client, triage)
print(json.dumps(result, indent=2))

{
  "model": "sap-rpt-1.6",
  "answers": [
    {
      "billing": {
        "type": "noul",
        "noul": 0.74
      },
      "tone": {
        "type": "choice",
        "choice": "angry",
        "probabilities": {
          "calm": 0.24,
          "angry": 0.46,
          "neutral": 0.3
        },
        "confidence": 0.46
      },
      "priority": {
        "type": "score",
        "score": 1.22,
        "legend": {
          "0": "can wait",
          "1": "this week",
          "2": "today"
        },
        "probabilities": {
          "0": 0.2,
          "1": 0.38,
          "2": 0.42
        },
        "confidence": 0.42
      }
    }
  ],
  "metadata": {
    "context_mode": "default",
    "num_columns": 5,
    "num_predictions": 3,
    "num_query_rows": 1,
    "num_rows": 6
  }
}


How the fields are derived from the RPT response:

* `noul`: the probability RPT assigns to the "yes" value.
* `choice`: the option with the highest probability, the full distribution, and its probability as `confidence`.
* `score`: `score` is the expected level (sum of level index times probability), e.g. 1.2 means "between *this week* and *today*, closer to *this week*"; `confidence` is the probability of the most likely level.

## 6. Encoding matters: `described` vs `plain`

`plain` writes the raw keys exactly like the Playground body (`billing`, `true`, `angry`, `"2"`). The comparison below runs both encodings on a few new messages. With only five examples the answers are fragile, which is exactly where the extra meaning helps.

In [6]:
new_messages = [
    "I was charged $99 instead of $49 on my last bill, please fix this!",
    "Love the new dashboard, great work team!",
    "Where can I download last year's invoices?",
    "This is the worst support ever, my account is locked AGAIN.",
]
rows = []
for encoding in ["plain", "described"]:
    answers = decide(client, triage, states=new_messages, encoding=encoding)["answers"]
    for message, answer in zip(new_messages, answers):
        rows.append({
            "encoding": encoding,
            "message": message,
            "billing (p)": answer["billing"]["noul"],
            "tone": f'{answer["tone"]["choice"]} ({answer["tone"]["confidence"]:.2f})',
            "priority (score)": answer["priority"]["score"],
        })
pd.DataFrame(rows).set_index(["message", "encoding"]).sort_index()

billing (p)  \
message                                                            encoding                 
I was charged $99 instead of $49 on my last bill, please fix this! described         0.74   
                                                                   plain             0.60   
Love the new dashboard, great work team!                           described         0.07   
                                                                   plain             0.05   
This is the worst support ever, my account is locked AGAIN.        described         0.43   
                                                                   plain             0.48   
Where can I download last year's invoices?                         described         0.15   
                                                                   plain             0.07   

                                                                                      tone  \
message                                                            encoding                  
I was charged $99 instead of $49 on my last bill, please fix this! described  angry (0.46)   
                                                                   plain       calm (0.45)   
Love the new dashboard, great work team!                           described   calm (0.96)   
                                                                   plain       calm (0.89)   
This is the worst support ever, my account is locked AGAIN.        described   calm (0.40)   
                                                                   plain       calm (0.40)   
Where can I download last year's invoices?                         described   calm (0.57)   
                                                                   plain       calm (0.45)   

                                                                              priority (score)  
message                                                            encoding                     
I was charged $99 instead of $49 on my last bill, please fix this! described              1.22  
                                                                   plain                  1.10  
Love the new dashboard, great work team!                           described              0.07  
                                                                   plain                  0.11  
This is the worst support ever, my account is locked AGAIN.        described              0.88  
                                                                   plain                  0.92  
Where can I download last year's invoices?                         described              0.57  
                                                                   plain                  0.61

Read the table critically. In our run `described` fixed the tone of the overcharge complaint (`angry` instead of `calm`) and raised its billing probability, but both encodings still call *"This is the worst support ever..."* `calm`: five examples are very little context, and none of them is an angry message about something other than billing. In production, collect a few hundred labeled examples per question that cover the variety you expect, and measure accuracy on a held-out set before choosing an encoding.

The cell above also showed **batching**: `states=[...]` answers up to 512 states in one `/predict` call (one query row each), while the Playground endpoint takes one state per request.

## 7. Structured states

A state can be a JSON object. Each key becomes its own column, which lets RPT use fields such as the channel or customer tier next to the text. Here: should a support ticket be escalated, and which team owns it?

Design the examples so that no single field gives the answer away. In a first draft every gold-tier ticket was escalated, and the model simply learned "gold means escalate" (probabilities of exactly 1.00 and 0.00). Below, escalation depends on the problem, not the tier.

In [7]:
ticket_routing = {
    "questions": {
        "escalate": {
            "type": "noul",
            "instructions": "Should this ticket be escalated to a supervisor?",
            "criteria": {"true": "Needs a supervisor now", "false": "Normal queue"},
        },
        "team": {
            "type": "choice",
            "instructions": "Which team should handle the ticket?",
            "criteria": {
                "billing": "Invoices, charges, refunds",
                "technical": "Errors, outages, access problems",
                "sales": "Pricing, upgrades, new contracts",
            },
        },
    },
    "examples": [
        {"state": {"channel": "phone", "customer_tier": "gold", "message": "Production system down since this morning."},
         "answers": {"escalate": True, "team": "technical"}},
        {"state": {"channel": "email", "customer_tier": "standard", "message": "Please resend my March invoice."},
         "answers": {"escalate": False, "team": "billing"}},
        {"state": {"channel": "chat", "customer_tier": "standard", "message": "What does the premium plan cost?"},
         "answers": {"escalate": False, "team": "sales"}},
        {"state": {"channel": "email", "customer_tier": "standard", "message": "Refund still missing after 6 weeks, final warning."},
         "answers": {"escalate": True, "team": "billing"}},
        {"state": {"channel": "chat", "customer_tier": "gold", "message": "I cannot log in after the password reset."},
         "answers": {"escalate": False, "team": "technical"}},
        {"state": {"channel": "phone", "customer_tier": "gold", "message": "We want to add 200 seats before quarter end."},
         "answers": {"escalate": False, "team": "sales"}},
        {"state": {"channel": "email", "customer_tier": "gold", "message": "Can you send a price list for additional licenses?"},
         "answers": {"escalate": False, "team": "sales"}},
        {"state": {"channel": "chat", "customer_tier": "standard", "message": "The app crashes for every user in our company."},
         "answers": {"escalate": True, "team": "technical"}},
    ],
}
tickets = [
    {"channel": "phone", "customer_tier": "gold", "message": "Nobody can access the portal, all users blocked."},
    {"channel": "email", "customer_tier": "standard", "message": "Can I get a quote for the enterprise edition?"},
]
print(pd.DataFrame(build_decision_payload(ticket_routing, tickets)["rows"]).to_string())
for ticket, answer in zip(tickets, decide(client, ticket_routing, states=tickets)["answers"]):
    print(f'\n{ticket["message"]}\n  escalate p={answer["escalate"]["noul"]:.2f} | team={answer["team"]["choice"]} {answer["team"]["probabilities"]}')

      row_id channel customer_tier                                             message Should this ticket be escalated to a supervisor?         Which team should handle the ticket?
0  example-0   phone          gold          Production system down since this morning.                      yes: Needs a supervisor now  technical: Errors, outages, access problems
1  example-1   email      standard                     Please resend my March invoice.                                 no: Normal queue          billing: Invoices, charges, refunds
2  example-2    chat      standard                    What does the premium plan cost?                                 no: Normal queue      sales: Pricing, upgrades, new contracts
3  example-3   email      standard  Refund still missing after 6 weeks, final warning.                      yes: Needs a supervisor now          billing: Invoices, charges, refunds
4  example-4    chat          gold           I cannot log in after the password reset.         


Nobody can access the portal, all users blocked.
  escalate p=0.15 | team=technical {'billing': 0.0, 'technical': 1.0, 'sales': 0.0}

Can I get a quote for the enterprise edition?
  escalate p=0.59 | team=sales {'billing': 0.34, 'technical': 0.24, 'sales': 0.42}


In our run both tickets are routed to the right team, but the escalation probabilities are the wrong way round (the outage gets a low value, the quote request a higher one). Eight examples are not enough for RPT to learn *escalation* from free text; with structured fields this small, more labeled history is the only real fix.

## 8. Validation before the call

`validate_decision_request` (run inside `build_decision_payload`) applies the Playground's documented limits and the label demonstration rule, so mistakes fail early with a readable message instead of a model error. Here the score level `"2"` (*today*) never appears in the examples:

In [8]:
broken = json.loads(json.dumps(triage))
for example in broken["examples"]:
    if example["answers"]["priority"] == "2":
        example["answers"]["priority"] = "1"
try:
    validate_decision_request(broken)
except ValueError as error:
    print("ValueError:", error)

ValueError: Question 'priority': options ['2'] never appear in the examples; RPT can only predict labels it has seen in context.


## Takeaways

* SAP AI Core has no decision endpoint; the Playground body is rejected by `/predict` (HTTP 422). A decision is still one RPT call once it is expressed as a table.
* Request every option with `top_k` equal to the number of options to get the full, calibrated distribution, and keep all questions in one joint call (as the Playground does) or split them if you need independent answers.
* Encoding is a modeling choice. Meaningful column names and descriptive values gave better answers in our small test; confirm on your own labeled holdout.
* Batch up to 512 states per call. `sap-rpt-1.6` handles up to 2,048 context rows (examples); beyond that use `sap-rpt-1.6-large`.
* Keep the request definition (questions, criteria, examples, encoding, model) with every stored decision so answers stay auditable.